# Session 6: cost per resolved ticket

Exploration only. Runs fifty tickets through the Session 5 loop and through the router on the local tools, and does the arithmetic at Northwind's volume. Needs a configured track; costs about a dollar.

In [ ]:
import asyncio, json, statistics
from pathlib import Path
from nw.agent.loop import run_agent
from nw.agent.northwind import build_registry
from nw.agent.router import route
from nw.config import settings
from nw.llm import LLMClient
from nw.llm.providers import make_provider

s = settings(); client = LLMClient(make_provider(s), settings=s)
registry = build_registry("local", accounts_path=Path("../data/accounts.json"), artifacts=Path("../artifacts"))
rows = [json.loads(l) for l in open("../data/tickets.jsonl")][:50]
print(len(rows), "tickets, priorities:", {p: sum(r["priority"] == p for r in rows) for p in ("P0", "P1", "P2", "P3")})

## Through the loop

In [ ]:
async def loop_all():
    out = []
    for r in rows:
        t = await run_agent(f"Ticket {r['ticket_id']} from account {r['account_id']}\nSubject: {r['subject']}\n\n{r['body']}", registry, client)
        out.append((r["priority"], t.cost_usd, t.n_steps))
    return out
loop = asyncio.run(loop_all())
print(f"loop: {statistics.mean(c for _, c, _ in loop):.4f} USD per ticket, {statistics.mean(n for _, _, n in loop):.1f} steps")

## Through the router

In [ ]:
async def route_all():
    out = []
    for r in rows:
        t = await route(r["ticket_id"], r["account_id"], r["subject"], r["body"], registry, client)
        out.append((r["priority"], t.cost_usd, t.agent))
    return out
routed = asyncio.run(route_all())
print(f"router: {statistics.mean(c for _, c, _ in routed):.4f} USD per ticket")
for p in ("P0", "P1", "P2", "P3"):
    sub = [c for pr, c, _ in routed if pr == p]
    if sub: print(f"  {p}: {statistics.mean(sub):.4f} USD, n={len(sub)}, agents={set(a for pr, _, a in routed if pr == p)}")

## At Northwind's volume

In [ ]:
per_quarter = 8000
for name, data in (("loop", loop), ("router", routed)):
    mean = statistics.mean(c for _, c, *_ in data)
    print(f"{name:7s} {mean:.4f} USD per ticket -> {mean * per_quarter:8.0f} USD per quarter")

## Questions to bring back

- Which priority band drives the saving, and is the router's P0 threshold where the on-call lead wants it?
- What would the number be if P1 also went to the Economy model, and what would it cost in adversarial-set passes?